In [1]:
# pip install -q langchain langchain-community langchain-groq transformers torch sarvamai faiss-cpu sentence-transformers ffmpeg-python pdflatex


In [2]:
import os
import json
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass
from enum import Enum
import logging
from pathlib import Path
import re
import sys
import shutil
import tempfile
import subprocess
from contextlib import contextmanager
import importlib.util
import time
import uuid
import torch
from datetime import datetime

In [3]:
# LangChain imports
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.memory import ConversationBufferMemory
from langchain.schema import BaseMessage, HumanMessage, AIMessage
from langchain.tools import BaseTool, tool
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader, DirectoryLoader
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [4]:
# TTS imports
try:
    from sarvamai import SarvamAI
    from sarvamai.play import save
    SARVAM_AVAILABLE = True
except ImportError:
    SARVAM_AVAILABLE = False

In [5]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [6]:
@dataclass
class AnimationRequest:
    """Structure for animation requests"""
    description: str
    duration: int  # in seconds
    style: str = "educational"
    complexity: str = "medium"
    
@dataclass
class SceneOutline:
    """Structure for scene planning"""
    scenes: List[Dict[str, Any]]
    timeline: Dict[str, float]
    visual_elements: List[str]
    transitions: List[str]

@dataclass
class StoryboardPlan:
    """Structure for storyboard planning"""
    keyframes: List[Dict[str, Any]]
    visual_flow: List[str]
    camera_movements: List[str]

@dataclass
class TechnicalPlan:
    """Structure for technical implementation"""
    manim_objects: List[str]
    animations: List[str]
    code_structure: Dict[str, Any]
    dependencies: List[str]

@dataclass
class NarrationPlan:
    """Structure for narration planning"""
    script: str
    timing: Dict[str, float]
    voice_settings: Dict[str, Any]

@dataclass
class IterationResult:
    """Store results from each iteration attempt"""
    iteration: int
    code: str
    error: Optional[str] = None
    success: bool = False
    error_type: Optional[str] = None
    fixes_applied: List[str] = None


In [7]:
class AgentType(Enum):
    MANAGER = "manager"
    PLANNER = "planner"
    CODE_GENERATOR = "code_generator"
    CODE_EXECUTOR = "code_executor"
    NARRATOR = "narrator"
    AUDIO_GENERATOR = "audio_generator"
    RAG_ROUTER = "rag_router"

In [8]:
class BaseAgent:
    """Base class for all agents"""
    
    def __init__(self, name: str, llm: ChatGroq, agent_type: AgentType):
        self.name = name
        self.llm = llm
        self.agent_type = agent_type
        logger.info(f"Initialized {self.name} ({agent_type.value})")
    
    def process(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        raise NotImplementedError("Each agent must implement process method")

In [9]:
class RAGRouterAgent(BaseAgent):
    """Agent responsible for RAG system and context retrieval (with Manim documentation)"""
    
    def __init__(self, llm: ChatGroq):
        super().__init__("RAG Router", llm, AgentType.RAG_ROUTER)
        self.setup_embeddings()
        self.setup_rag_system()
    
    def setup_embeddings(self):
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        
    def setup_rag_system(self):
        docs_content = [
            "Manim Scene: A Scene is the main container for animations. Use Scene class to create animations.",
            "Manim Mobject: Mathematical objects (Mobject) are the building blocks of Manim animations.",
            "Manim Animations: Use Create(), Write(), Transform(), FadeIn(), FadeOut() for basic animations.",
            "Manim Camera: Control camera with self.camera.frame to zoom, pan, and rotate views.",
            "Manim Text: Use Text(), MathTex(), Tex() for displaying text and mathematical expressions.",
            "Manim Shapes: Circle(), Square(), Rectangle(), Line() for basic geometric shapes.",
            "Manim Colors: Use Color palette like BLUE, RED, GREEN, YELLOW for coloring objects.",
            "Manim Positioning: Use .move_to(), .shift(), .next_to() for positioning objects.",
            "Manim Grouping: Use Group() to group multiple objects together.",
            "Manim Timing: Use self.play() with run_time parameter to control animation timing.",
            "Manim Indentation: Always use 4 spaces for indentation, never tabs.",
            "Manim Imports: Use 'from manim import *' for all Manim objects and animations.",
            "Manim Variables: Initialize all variables before using them in animations."
        ]
        
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
        docs = text_splitter.create_documents(docs_content)
        
        self.vectorstore = FAISS.from_documents(docs, self.embeddings)
        self.retriever = self.vectorstore.as_retriever(search_kwargs={"k": 5})
        
    def process(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        query = input_data.get('query', '')
        relevant_docs = self.retriever.get_relevant_documents(query)
        context = "\n".join([doc.page_content for doc in relevant_docs])
        
        return {
            "context": context,
            "relevant_documents": relevant_docs
        }

In [10]:
class PlannerAgent(BaseAgent):
    """Agent responsible for creating scene outlines and storyboards"""
    
    def __init__(self, llm: ChatGroq, rag_agent: RAGRouterAgent):
        super().__init__("Scene Planner", llm, AgentType.PLANNER)
        self.rag_agent = rag_agent
    
    def process(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        request = input_data['request']
        
        logger.info(f"Creating scene outline for: {request.description}")
        
        rag_result = self.rag_agent.process({'query': request.description})
        context = rag_result['context']
        
        scene_prompt = ChatPromptTemplate.from_messages([
            ("system", f"You are an expert animation director. Create detailed scene outlines for educational {request.style} animations."),
            ("human", f"""
            Create a detailed scene outline for a {request.duration}-second Manim animation about: {request.description}
            
            Break it down into:
            1. Scene progression (what happens when)
            2. Key visual elements needed
            3. Timing for each scene (in seconds)
            4. Transitions between scenes
            5. Make sure the text doesn't overlap 
            
            Style: {request.style}
            Complexity: {request.complexity}
            
            Relevant Manim concepts:
            {context}
            
            Provide a clear, structured outline with specific timings.
            """)
        ])
        
        chain = scene_prompt | self.llm
        response = chain.invoke({})
        
        return {
            "scene_outline": response.content,
            "context": context
        }

In [12]:
class CodeGeneratorAgent(BaseAgent):
    """Agent responsible for generating Manim code"""
    
    def __init__(self, llm: ChatGroq, rag_agent: RAGRouterAgent):
        super().__init__("Code Generator", llm, AgentType.CODE_GENERATOR)
        self.rag_agent = rag_agent
        
        # error patterns for debugging
        self.error_patterns = {
            'indentation': r'IndentationError|unexpected indent|expected an indented block',
            'syntax': r'SyntaxError|invalid syntax|unexpected token',
            'unbound_local': r'UnboundLocalError|local variable.*referenced before assignment',
            'name_error': r'NameError|name.*is not defined',
            'import_error': r'ImportError|ModuleNotFoundError|No module named',
            'attribute_error': r'AttributeError|has no attribute',
            'type_error': r'TypeError|unsupported operand type|takes.*positional arguments',
            'manim_specific': r'Scene.*not found|construct.*method|Animation.*error'
        }
        
        self.debug_templates = {
            'indentation': """
The following Manim code has an IndentationError:

CODE:
{code}

ERROR:
{error}

PREVIOUS ATTEMPTS:
{previous_attempts}

Please fix the indentation issues and return only the corrected Python code. Make sure to:
1. Use exactly 4 spaces for each indentation level
2. Ensure proper class and method indentation
3. Check that all code blocks are properly indented
4. Verify that there are no mixing of tabs and spaces

Fixed code:
""",
            'general': """
The following Manim code has an error:

CODE:
{code}

ERROR:
{error}

PREVIOUS ATTEMPTS:
{previous_attempts}

Please analyze the error carefully and fix the code. Return only the corrected Python code. Make sure to:
1. Fix the specific error mentioned
2. Use proper Manim Community v0.17+ syntax
3. Include all necessary imports
4. Ensure the Scene class is properly defined
5. Avoid making the same mistakes from previous attempts

Fixed code:
"""
        }
    
    def process(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        request = input_data['request']
        scene_outline = input_data.get('scene_outline', '')
        
        logger.info(f"Generating Manim code for: {request.description}")
        
        rag_result = self.rag_agent.process({'query': request.description})
        context = rag_result['context']
        
        code_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert Manim developer. Generate clean, executable Python code for educational animations. 

CRITICAL RULES:
1. Always use exactly 4 spaces for indentation (never tabs)
2. Always include 'from manim import *' at the top
3. Always implement the construct method
4. Initialize all variables before using them
5. Use proper Manim Community v0.17+ syntax
6. Test your code mentally before outputting

TEMPLATE TO FOLLOW:
```python
from manim import *

class AnimationScene(Scene):
    def construct(self):
        # Initialize all objects first
        # Then animate them
        # Use self.play() for animations
        # Use self.wait() for pauses
        pass
```"""),
            ("human", f"""
Generate complete, executable Manim Python code for: {request.description}

Requirements:
- Duration: {request.duration} seconds
- Style: {request.style}
- Complexity: {request.complexity}

Scene Outline:
{scene_outline}

Relevant Manim concepts:
{context}

Return ONLY the Python code, properly indented with 4 spaces, no explanations.
""")
        ])
        
        chain = code_prompt | self.llm
        response = chain.invoke({})
        
        code = self.extract_code_from_response(response.content)
        if not code:
            code = response.content.strip()
        
        return {
            "code": code,
            "raw_response": response.content
        }
    
    def extract_code_from_response(self, response: str) -> str:
        # try to extract code from markdown blocks
        code_blocks = re.findall(r'```python\n(.*?)\n```', response, re.DOTALL)
        if code_blocks:
            return code_blocks[0].strip()
        
        # try to extract code from any code blocks
        code_blocks = re.findall(r'```\n(.*?)\n```', response, re.DOTALL)
        if code_blocks:
            return code_blocks[0].strip()
        
        # return the whole response if it looks like code
        if 'class' in response and 'def construct' in response:
            return response.strip()
        
        return ""
    
    def identify_error_type(self, error_message: str) -> str:
        for error_type, pattern in self.error_patterns.items():
            if re.search(pattern, error_message, re.IGNORECASE):
                return error_type
        return 'general'
    
    def debug_code(self, code: str, error: str, previous_attempts: List[IterationResult] = None) -> str:
        error_type = self.identify_error_type(error)
        template = self.debug_templates.get(error_type, self.debug_templates['general'])
        
        previous_attempts_text = self.format_previous_attempts(previous_attempts or [])
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert Python and Manim developer. Fix the code errors and provide a corrected version."),
            ("human", template.format(
                code=code,
                error=error,
                previous_attempts=previous_attempts_text
            ))
        ])
        
        chain = prompt | self.llm
        response = chain.invoke({})
        
        return self.extract_code_from_response(response.content)
    
    def format_previous_attempts(self, attempts: List[IterationResult]) -> str:
        if not attempts:
            return "No previous attempts."
        
        formatted = []
        for attempt in attempts:
            formatted.append(f"Attempt {attempt.iteration}:")
            formatted.append(f"  Error Type: {attempt.error_type}")
            formatted.append(f"  Error: {attempt.error}")
            formatted.append("")
        
        return "\n".join(formatted)

In [13]:
class CodeExecutorAgent(BaseAgent):
    """Agent responsible for executing and debugging Manim code"""
    
    def __init__(self, llm: ChatGroq, code_generator: CodeGeneratorAgent, output_dir: str):
        super().__init__("Code Executor", llm, AgentType.CODE_EXECUTOR)
        self.code_generator = code_generator
        self.output_dir = output_dir
        self.max_iterations = 5
    
    def process(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        # execute Manim code with iterative debugging
        code = input_data['code']
        
        logger.info(f"Executing Manim code")
        
        # first attempt
        initial_result = self._execute_code_once(code)
        
        if initial_result['success']:
            return initial_result
        
        logger.info("Initial execution failed, starting iterative debugging...")
        debug_result = self.debug_code_iteratively(code, initial_result['error'])
        
        if debug_result.success:
            logger.info("Code fixed through iterative debugging")
            return self._execute_code_once(debug_result.code)
        else:
            logger.error("Could not fix code through iterative debugging")
            return {
                "success": False,
                "error": f"Code could not be fixed after {self.max_iterations} iterations. Last error: {debug_result.error}",
                "debug_attempts": debug_result.iteration
            }
    
    def debug_code_iteratively(self, initial_code: str, initial_error: str) -> IterationResult:
        attempts = []
        current_code = initial_code
        
        for iteration in range(1, self.max_iterations + 1):
            logger.info(f"Debug iteration {iteration}/{self.max_iterations}")
            
            execution_result = self.test_code_execution(current_code)
            
            if execution_result['success']:
                logger.info(f"Code fixed successfully in iteration {iteration}")
                return IterationResult(
                    iteration=iteration,
                    code=current_code,
                    success=True
                )
            
            error_message = execution_result['error']
            error_type = self.code_generator.identify_error_type(error_message)
            
            # record this attempt
            attempt = IterationResult(
                iteration=iteration,
                code=current_code,
                error=error_message,
                error_type=error_type
            )
            attempts.append(attempt)
            
            try:
                new_code = self.code_generator.debug_code(current_code, error_message, attempts[:-1])
                if new_code and new_code != current_code:
                    current_code = new_code
                    logger.info(f"Applied debugging fixes for {error_type} error")
                else:
                    logger.warning(f"No different solution provided in iteration {iteration}")
                    if iteration > 2:
                        break
                        
            except Exception as e:
                logger.error(f"Error in debugging iteration {iteration}: {e}")
                break
        
        logger.warning(f"Could not fix code after {self.max_iterations} iterations")
        return IterationResult(
            iteration=self.max_iterations,
            code=current_code,
            error=f"Could not fix after {self.max_iterations} attempts",
            success=False
        )
    
    def test_code_execution(self, code: str) -> Dict[str, Any]:
        with self._create_temp_dir() as temp_dir:
            try:
                scene_name = self._extract_scene_name(code)
                if not scene_name:
                    return {
                        "success": False,
                        "error": "Could not find a valid Scene class in the code"
                    }
                
                script_path = os.path.join(temp_dir, "test_scene.py")
                with open(script_path, "w") as f:
                    f.write(code)
                
                sys.path.insert(0, temp_dir)
                
                try:
                    spec = importlib.util.spec_from_file_location("test_scene", script_path)
                    module = importlib.util.module_from_spec(spec)
                    spec.loader.exec_module(module)
                    
                    if hasattr(module, scene_name):
                        scene_class = getattr(module, scene_name)
                        if hasattr(scene_class, 'construct'):
                            return {"success": True}
                        else:
                            return {"success": False, "error": f"Scene class {scene_name} missing construct method"}
                    else:
                        return {"success": False, "error": f"Scene class {scene_name} not found"}
                        
                except Exception as e:
                    return {"success": False, "error": str(e)}
                finally:
                    sys.path.remove(temp_dir)
                    
            except Exception as e:
                return {"success": False, "error": str(e)}
    
    def _execute_code_once(self, code: str) -> Dict[str, Any]:
        with self._create_temp_dir() as temp_dir:
            try:
                scene_name = self._extract_scene_name(code)
                if not scene_name:
                    raise Exception("Could not find a valid Scene class in the code")
                
                script_path = os.path.join(temp_dir, "scene.py")
                with open(script_path, "w") as f:
                    f.write(code)
                
                logger.info(f"Executing scene: {scene_name}")
                
                result = subprocess.run(
                    [
                        "python",
                        "-m",
                        "manim",
                        "-ql",
                        "--media_dir", temp_dir,
                        script_path,
                        scene_name
                    ],
                    capture_output=True,
                    text=True,
                    cwd=temp_dir
                )
                
                if result.returncode != 0:
                    return {
                        "success": False,
                        "error": result.stderr,
                        "stdout": result.stdout,
                        "scene_name": scene_name
                    }
                
                video_files = []
                for root, dirs, files in os.walk(temp_dir):
                    for file in files:
                        if file.endswith('.mp4'):
                            video_files.append(os.path.join(root, file))
                
                if not video_files:
                    return {
                        "success": False,
                        "error": f"No video files generated",
                        "scene_name": scene_name
                    }
                
                video_path = video_files[0]
                output_filename = f"output_{scene_name}.mp4"
                output_path = os.path.join(self.output_dir, output_filename)
                
                shutil.copy(video_path, output_path)
                
                return {
                    "success": True,
                    "video_path": output_path,
                    "scene_name": scene_name,
                    "stdout": result.stdout
                }
                    
            except Exception as e:
                return {
                    "success": False,
                    "error": str(e),
                    "scene_name": scene_name if 'scene_name' in locals() else None
                }
    
    @contextmanager
    def _create_temp_dir(self):
        """Create a temporary directory context manager"""
        temp_dir = tempfile.mkdtemp()
        try:
            yield temp_dir
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)

    def _extract_scene_name(self, code: str) -> Optional[str]:
        pattern = r'class\s+([A-Za-z_][A-Za-z0-9_]*)\s*\([^)]*Scene[^)]*\):'
        matches = re.findall(pattern, code)
        if matches:
            return matches[0]
        
        pattern = r'class\s+([A-Za-z_][A-Za-z0-9_]*Scene[A-Za-z0-9_]*)\s*\([^)]*\):'
        matches = re.findall(pattern, code)
        if matches:
            return matches[0]
        
        return None

In [14]:
class NarratorAgent(BaseAgent):
    """Agent responsible for creating narration scripts"""
    
    def __init__(self, llm: ChatGroq):
        super().__init__("Narrator", llm, AgentType.NARRATOR)
    
    def process(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        request = input_data['request']
        scene_outline = input_data.get('scene_outline', '')
        
        logger.info(f"Generating narration script for: {request.description}")
        
        script_prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a professional educational content narrator. Create engaging, clear narration scripts."),
            ("human", f"""
            Create a narration script for a {request.duration}-second animation about: {request.description}
            
            Requirements:
            1. Clear, engaging narration matching visual elements
            2. Should be exactly {request.duration} seconds when spoken
            3. Educational and informative tone
            4. Natural speech patterns
            
            Scene Outline:
            {scene_outline}
            
            Write only the script text, no formatting.
            """)
        ])
        
        chain = script_prompt | self.llm
        response = chain.invoke({})
        
        return {
            "script": response.content.strip()
        }

In [15]:
class AudioGeneratorAgent(BaseAgent):
    """Agent responsible for generating audio narration"""
    
    def __init__(self, llm: ChatGroq, sarvam_api_key: str = None):
        super().__init__("Audio Generator", llm, AgentType.AUDIO_GENERATOR)
        self.sarvam_api_key = sarvam_api_key
        self.setup_tts()
    
    def setup_tts(self):
        try:
            if SARVAM_AVAILABLE and self.sarvam_api_key:
                self.tts_client = SarvamAI(api_subscription_key=self.sarvam_api_key)
                logger.info("TTS client initialized")
            else:
                self.tts_client = None
                logger.warning("TTS not available - API key missing or Sarvam not installed")
        except Exception as e:
            logger.error(f"Failed to initialize TTS: {e}")
            self.tts_client = None
    
    def process(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        script = input_data['script']
        output_dir = input_data['output_dir']
        
        logger.info(f"Generating audio narration")
        
        try:
            if not self.tts_client:
                return {
                    "success": False,
                    "error": "TTS client not available"
                }
            
            output_file = os.path.join(output_dir, "narration.wav")
            audio = self.tts_client.text_to_speech.convert(
                target_language_code="en-IN",
                text=script,
                model="bulbul:v2",
                speaker="anushka"
            )
            save(audio, output_file)
            
            return {
                "success": True,
                "audio_file": output_file
            }
        except Exception as e:
            return {
                "success": False,
                "error": str(e)
            }
    
    def combine_audio_video(self, video_file: str, audio_file: str, output_dir: str) -> Dict[str, Any]:
        try:
            output_file = os.path.join(output_dir, "final_animation.mp4")
            cmd = [
                "ffmpeg", "-y",
                "-i", video_file,
                "-i", audio_file,
                "-c:v", "copy",
                "-c:a", "aac",
                "-shortest",
                output_file
            ]
            
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0:
                return {
                    "success": True,
                    "final_video": output_file
                }
            else:
                return {
                    "success": False,
                    "error": result.stderr
                }
        except Exception as e:
            return {
                "success": False,
                "error": str(e)
            }

In [16]:
class ManagerAgent(BaseAgent):
    """Manager agent that coordinates all other agents"""
    
    def __init__(self, llm: ChatGroq, sarvam_api_key: str = None, output_dir: str = "/kaggle/working"):
        super().__init__("Manager", llm, AgentType.MANAGER)
        self.output_dir = output_dir
        
        self.rag_agent = RAGRouterAgent(llm)
        self.planner_agent = PlannerAgent(llm, self.rag_agent)
        self.code_generator_agent = CodeGeneratorAgent(llm, self.rag_agent)
        self.code_executor_agent = CodeExecutorAgent(llm, self.code_generator_agent, output_dir)
        self.narrator_agent = NarratorAgent(llm)
        self.audio_generator_agent = AudioGeneratorAgent(llm, sarvam_api_key)
        
        logger.info("Manager Agent initialized with all sub-agents")
    
    def process(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        request = input_data['request']
        
        logger.info(f"Starting animation generation for: {request.description}")
        
        results = {}
        
        try:
            # Step 1: Create scene outline
            print("\n" + "="*60)
            print("STEP 1: CREATING SCENE OUTLINE")
            print("="*60)
            
            planner_result = self.planner_agent.process({'request': request})
            results["scene_outline"] = planner_result["scene_outline"]
            
            print("SCENE OUTLINE:")
            print("-" * 40)
            print(planner_result["scene_outline"])
            
            # Step 2: Generate Manim code
            print("\n" + "="*60)
            print("STEP 2: GENERATING MANIM CODE")
            print("="*60)
            
            code_gen_result = self.code_generator_agent.process({
                'request': request,
                'scene_outline': planner_result["scene_outline"]
            })
            results["code"] = code_gen_result["code"]
            
            print("GENERATED MANIM CODE:")
            print("-" * 40)
            print(code_gen_result["code"])
            
            # Step 3: Execute Manim code
            print("\n" + "="*60)
            print("STEP 3: EXECUTING MANIM CODE")
            print("="*60)
            
            execution_result = self.code_executor_agent.process({
                'code': code_gen_result["code"]
            })
            results["execution_result"] = execution_result
            
            print("EXECUTION RESULTS:")
            print("-" * 40)
            print(f"Success: {execution_result.get('success', False)}")
            
            if execution_result.get('success'):
                print(f"Video Path: {execution_result.get('video_path')}")
                print(f"Scene Name: {execution_result.get('scene_name')}")
            else:
                print(f"Error: {execution_result.get('error', 'Unknown error')}")
                if 'debug_attempts' in execution_result:
                    print(f"Debug attempts made: {execution_result['debug_attempts']}")
            
            # Step 4: Generate narration script
            print("\n" + "="*60)
            print("STEP 4: GENERATING NARRATION SCRIPT")
            print("="*60)
            
            narrator_result = self.narrator_agent.process({
                'request': request,
                'scene_outline': planner_result["scene_outline"]
            })
            results["narration"] = narrator_result["script"]
            
            print("NARRATION SCRIPT:")
            print("-" * 40)
            print(narrator_result["script"])
            
            # Step 5: Generate audio (if TTS is available)
            print("\n" + "="*60)
            print("STEP 5: GENERATING AUDIO NARRATION")
            print("="*60)
            
            if SARVAM_AVAILABLE and self.audio_generator_agent.sarvam_api_key:
                audio_result = self.audio_generator_agent.process({
                    'script': narrator_result["script"],
                    'output_dir': self.output_dir
                })
                results["audio_result"] = audio_result
                
                print("AUDIO GENERATION RESULTS:")
                print("-" * 40)
                print(f"Success: {audio_result.get('success', False)}")
                
                if audio_result.get('success'):
                    print(f"Audio File: {audio_result.get('audio_file')}")
                    
                    # Step 6: Combine audio and video
                    if execution_result.get('success'):
                        print("\n" + "="*60)
                        print("STEP 6: COMBINING AUDIO AND VIDEO")
                        print("="*60)
                        
                        combine_result = self.audio_generator_agent.combine_audio_video(
                            execution_result["video_path"],
                            audio_result["audio_file"],
                            self.output_dir
                        )
                        results["final_video"] = combine_result
                        
                        print("FINAL VIDEO RESULTS:")
                        print("-" * 40)
                        print(f"Success: {combine_result.get('success', False)}")
                        if combine_result.get('success'):
                            print(f"Final Video: {combine_result.get('final_video')}")
                        else:
                            print(f"Error: {combine_result.get('error')}")
                else:
                    print(f"Audio Error: {audio_result.get('error')}")
            else:
                print("Audio generation skipped - TTS not available or API key not provided")
                results["audio_result"] = {"success": False, "error": "TTS not available"}
            
            print("\n" + "="*60)
            print("FINAL SUMMARY")
            print("="*60)
            
            summary = {
                "animation_request": request.description,
                "duration": request.duration,
                "style": request.style,
                "scene_planning": "✅ Completed",
                "code_generation": "✅ Completed",
                "code_execution": "✅ Success" if execution_result.get('success') else "❌ Failed",
                "narration": "✅ Completed",
                "audio_generation": "✅ Success" if results.get("audio_result", {}).get('success') else "❌ Failed/Skipped",
                "final_video": "✅ Success" if results.get("final_video", {}).get('success') else "❌ Failed/Skipped"
            }
            
            for key, value in summary.items():
                if key == "animation_request":
                    print(f" {key.replace('_', ' ').title()}: {value}")
                elif key in ["duration", "style"]:
                    print(f" {key.replace('_', ' ').title()}: {value}")
                else:
                    print(f" {key.replace('_', ' ').title()}: {value}")
            
            results["summary"] = summary
            return results
            
        except Exception as e:
            logger.error(f"Error in animation generation pipeline: {e}")
            return {
                "success": False,
                "error": str(e),
                "partial_results": results
            }



In [17]:
class AnimationPipeline:
    """Main pipeline class for the multi-agent animation system"""
    
    def __init__(self, groq_api_key: str, sarvam_api_key: str = None, output_dir: str = "/kaggle/working"):
        self.groq_api_key = groq_api_key
        self.sarvam_api_key = sarvam_api_key
        self.output_dir = output_dir
        
        os.makedirs(self.output_dir, exist_ok=True)
        
        self.llm = ChatGroq(
            groq_api_key=groq_api_key,
            model_name="llama3-70b-8192",
            temperature=0.1
        )
        
        self.manager = ManagerAgent(self.llm, sarvam_api_key, output_dir)
        
        logger.info("Animation Pipeline initialized successfully")
    
    def create_animation(self, description: str, duration: int = 10, style: str = "educational", complexity: str = "medium") -> Dict[str, Any]:
        """Create an animation based on the given parameters"""
        request = AnimationRequest(
            description=description,
            duration=duration,
            style=style,
            complexity=complexity
        )
        
        logger.info(f"Creating animation: {description}")
        
        results = self.manager.process({'request': request})
        
        return results
    
    def get_available_outputs(self) -> List[str]:
        outputs = []
        for file in os.listdir(self.output_dir):
            if file.endswith(('.mp4', '.wav', '.py')):
                outputs.append(os.path.join(self.output_dir, file))
        return outputs

In [ ]:
def test_pipeline():
    GROQ_API_KEY = "insert_groq_api"
    SARVAM_API_KEY = "insert_sarvam_api"  # Optional
    
    try:
        pipeline = AnimationPipeline(
            groq_api_key=GROQ_API_KEY,
            sarvam_api_key=SARVAM_API_KEY,
            output_dir="./outputs"
        )
        
        results = pipeline.create_animation(
            description="Expand (a+b)^2 without using any squares",
            duration=15,
            style="educational",
            complexity="medium"
        )
        
        print("\nPIPELINE TEST COMPLETED")
        print("=" * 50)
        print(f"Results: {results.get('summary', {})}")
        
        outputs = pipeline.get_available_outputs()
        if outputs:
            print(f"\nAvailable output files:")
            for output in outputs:
                print(f"   - {output}")
        
        return results
        
    except Exception as e:
        logger.error(f"Pipeline test failed: {e}")
        return {"success": False, "error": str(e)}

In [19]:
def create_custom_animation(groq_api_key: str, description: str, duration: int = 10, 
                           sarvam_api_key: str = None, output_dir: str = "./outputs"):
    try:
        pipeline = AnimationPipeline(
            groq_api_key=groq_api_key,
            sarvam_api_key=sarvam_api_key,
            output_dir=output_dir
        )
        
        results = pipeline.create_animation(
            description=description,
            duration=duration
        )
        
        return results
        
    except Exception as e:
        logger.error(f"Custom animation creation failed: {e}")
        return {"success": False, "error": str(e)}


In [20]:
test_results = test_pipeline()

INFO:__main__:Initialized Manager (manager)
INFO:__main__:Initialized RAG Router (rag_router)
C:\Users\Acer\AppData\Local\Temp\ipykernel_33424\1919461586.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:faiss.loader:Loading faiss with AVX512 support.
INFO:faiss.loader:Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
INFO:faiss.loader:Loading fa


STEP 1: CREATING SCENE OUTLINE


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Generating Manim code for: Expand (a+b)^2 without using any squares


SCENE OUTLINE:
----------------------------------------
Here is a detailed scene outline for a 20-second Manim animation about expanding (a+b)^2 without using any squares:

**Scene Outline:**

**Scene 1: Introduction (0s-2s)**

* Scene progression: The animation starts with a blank screen, and the equation (a+b)^2 appears on screen.
* Key visual elements needed: MathTex object for the equation (a+b)^2
* Timing: 2 seconds
* Transitions: FadeIn() animation for the equation

**Scene 2: Expand the Equation (2s-6s)**

* Scene progression: The equation (a+b)^2 is expanded to a^2 + 2ab + b^2, with each term appearing one by one.
* Key visual elements needed: MathTex objects for each term (a^2, 2ab, b^2)
* Timing: 4 seconds
* Transitions: Create() animation for each term, with a slight delay between each term to create a sense of building up the equation

**Scene 3: Visual Representation (6s-12s)**

* Scene progression: A visual representation of the expansion is shown, using rectangles to rep

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Executing Manim code
INFO:__main__:Executing scene: ExpandEquation


...

EXECUTION RESULTS:
----------------------------------------
Success: True
Video Path: ./outputs\output_ExpandEquation.mp4
Scene Name: ExpandEquation

STEP 4: GENERATING NARRATION SCRIPT


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:Generating audio narration


NARRATION SCRIPT:
----------------------------------------
Here is the narration script for the 20-second animation:

"Let's expand the equation a plus b squared. To do this, we'll follow the order of operations and multiply the binomial by itself. First, we have a squared, then two a b, and finally b squared. Now, let's visualize this expansion. We can represent each term as a rectangle, with a squared in blue, two a b in green, and b squared in red. Notice how each term contributes to the overall expansion. Let's highlight each term individually. Here's a squared, then two a b, and finally b squared. And there you have it, the expanded equation a plus b squared, without using any squares."

STEP 5: GENERATING AUDIO NARRATION


INFO:httpx:HTTP Request: POST https://api.sarvam.ai/text-to-speech "HTTP/1.1 200 OK"


AUDIO GENERATION RESULTS:
----------------------------------------
Success: True
Audio File: ./outputs\narration.wav

STEP 6: COMBINING AUDIO AND VIDEO
FINAL VIDEO RESULTS:
----------------------------------------
Success: True
Final Video: ./outputs\final_animation.mp4

FINAL SUMMARY
 Animation Request: Expand (a+b)^2 without using any squares
 Duration: 20
 Style: educational
 Scene Planning: ✅ Completed
 Code Generation: ✅ Completed
 Code Execution: ✅ Success
 Narration: ✅ Completed
 Audio Generation: ✅ Success
 Final Video: ✅ Success

PIPELINE TEST COMPLETED
Results: {'animation_request': 'Expand (a+b)^2 without using any squares', 'duration': 20, 'style': 'educational', 'scene_planning': '✅ Completed', 'code_generation': '✅ Completed', 'code_execution': '✅ Success', 'narration': '✅ Completed', 'audio_generation': '✅ Success', 'final_video': '✅ Success'}

Available output files:
   - ./outputs\final_animation.mp4
   - ./outputs\narration.wav
   - ./outputs\output_ExpandEquation.mp4